# Mapa de rejeição

Agregue achados de lotes sintéticos por prestador e mês. Identificadores CPF são pseudonimizados com um segredo local.

In [1]:
import json
import os
import subprocess
import sys
from copy import deepcopy
from datetime import date
from pathlib import Path
from tempfile import TemporaryDirectory

from tiss import examples, parse_lote, rejection_map

base = parse_lote(examples.load("lote-valido"))
for guia in base["guias"]:
    guia["procedimentos_executados"][0]["data_execucao"] = "2026-10-21"

lotes = [deepcopy(base) for _ in range(3)]
lotes[0]["prestador"] = {"codigo_prestador_na_operadora": "CLINICA-A"}
lotes[1]["prestador"] = {"cnpj": "11222333000181"}
lotes[1]["guias"][0]["procedimentos_executados"][0]["valor_total"] = 0
lotes[2]["prestador"] = {"cpf": "123.456.789-09"}

mapa = rejection_map(
    lotes,
    segredo_pseudonimo="s" * 16,
    at=date(2026, 10, 21),
)
linhas_resumo = [
    (linha["provider"], linha["code"], linha["guias"], linha["affected_amount"])
    for linha in mapa["rows"]
]
linhas_resumo

[('cnpj:11222333000181', 'audit.valor_nao_positivo', 1, '0.00'),
 ('cnpj:11222333000181', 'item.valor_divergente', 1, '0.00'),
 ('cnpj:11222333000181', 'item.valor_zerado', 1, '100.00'),
 ('cnpj:11222333000181', 'lote.valor_total_geral', 1, '100.00'),
 ('cpf:hmac:4bc3e26011e3d2415208e751c83b6f999fd3d8e9bd0ee74386a0b33e43de0f4e',
  'xsd.padrao',
  2,
  '200.00')]

A página HTML é estática e não depende de serviços externos. A CLI também lê XML e JSON recursivamente: `python -m tiss mapa ./lotes --json mapa.json --html mapa.html`.

In [2]:
with TemporaryDirectory() as pasta_temporaria:
    raiz = Path(pasta_temporaria)
    pasta_lotes = raiz / "lotes"
    pasta_lotes.mkdir()
    for indice, lote in enumerate(lotes, start=1):
        (pasta_lotes / f"lote-{indice}.json").write_text(
            json.dumps(lote, ensure_ascii=False), encoding="utf-8"
        )
    pagina = raiz / "mapa.html"
    subprocess.run(
        [sys.executable, "-m", "tiss", "mapa", str(pasta_lotes), "--html", str(pagina)],
        env={
            **os.environ,
            "TISS_SEGREDO_PSEUDONIMO": "s" * 16,
        },
        check=True,
        capture_output=True,
    )
    resultado_html = (
        pagina.is_file(),
        "<script" not in pagina.read_text(encoding="utf-8").lower(),
    )
resultado_html

(True, True)